# 00 — Opening: neuronal equations as a biophysics ladder

This notebook contextually merges the original `neural_simulations_tutorial.ipynb` and the TFNE-Izhikevich notebook into a book-style opening chapter.  It is a map of the model families used throughout the tutorial set.

Scientific status: **tutorial exploratory, not biological truth**.


In [1]:
import os, sys, math, json
from pathlib import Path
import numpy as np

SMOKE_MODE = True
SEED = 7
np.random.seed(SEED)

# Allow running from repo root or from the tutorials directory.
ROOT = Path.cwd()
if not (ROOT / "src" / "jbiophysic").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
print("ROOT", ROOT)
print("SMOKE_MODE", SMOKE_MODE, "SEED", SEED)


ROOT /mnt/data/jbio_improve/jbiophysic-main
SMOKE_MODE True SEED 7


## Equation families

| Family | State variables | Native units | Timescale | Best use | Failure mode |
|---|---|---|---|---|---|
| Point process | spike times | probability / Hz | event-based | spike-train statistics | hides voltage and currents |
| Rate code | population rate | Hz / arbitrary | ms-s | low-dimensional population dynamics | loses spike timing |
| Discrete linear | x[t] | arbitrary | sampled | stability, filtering, control | too linear for spiking |
| LIF | V | mV | ms | threshold/reset intuition | no active conductances |
| QIF/EIF | V | mV | ms | spike onset nonlinearity | phenomenological |
| Izhikevich | v, u | mV/ms-scaled, current-like I | ms | fast spiking network exploration | I is not SI current |
| Hodgkin-Huxley | V, m, h, n | mV, ms, uA/cm² | sub-ms to ms | current-balance biophysics | stiff and parameter-heavy |
| Cable/Jaxley | compartment voltages/gates | biophysical | sub-ms to ms | dendrites/morphology | costly at scale |
| TFNE | phi_i, phi_e, V_m, J, CSD | SI | field/time-domain | CSD/LFP forward modeling | invalid if used without gauges, sources, tolerances |


In [2]:
import pandas as pd
rows = [
    ("Izhikevich", "v,u", "mV/ms-scaled", "fast network scaffold", "calibrate before TFNE"),
    ("HH", "V,m,h,n", "mV/ms/uA/cm^2", "membrane current balance", "small dt needed"),
    ("TFNE", "phi_i,phi_e,V_m,J,CSD", "SI", "forward field/CSD/LFP", "requires gauge/source conservation"),
]
df = pd.DataFrame(rows, columns=["model", "state", "units", "role", "guardrail"])
display(df)


,model,state,units,role,guardrail
0,Izhikevich,"v,u",mV/ms-scaled,fast network scaffold,calibrate before TFNE
1,HH,"V,m,h,n",mV/ms/uA/cm^2,membrane current balance,small dt needed
2,TFNE,"phi_i,phi_e,V_m,J,CSD",SI,forward field/CSD/LFP,requires gauge/source conservation


## Minimal LIF as the linear-to-threshold bridge


In [3]:
def simulate_lif(I=1.3, tau_ms=20.0, R=20.0, V_rest=-65.0, V_th=-50.0, V_reset=-65.0, dt_ms=0.1, T_ms=200.0):
    steps = int(T_ms / dt_ms)
    V = np.empty(steps)
    spikes = np.zeros(steps, dtype=bool)
    v = V_rest
    for t in range(steps):
        dv = (-(v - V_rest) + R * I) / tau_ms
        v += dt_ms * dv
        if v >= V_th:
            spikes[t] = True
            v = V_reset
        V[t] = v
    return V, spikes
V, sp = simulate_lif()
print({"finite": bool(np.isfinite(V).all()), "n_spikes": int(sp.sum()), "V_range_mV": (float(V.min()), float(V.max()))})


{'finite': True, 'n_spikes': 11, 'V_range_mV': (-65.0, -50.033674185633075)}


## Book doctrine

Every later notebook follows the same rule: state the equation, declare units, simulate, validate finite outputs, and mark the result as exploratory unless a validation benchmark supports a stronger claim.
